# Backpropagation — Step by Step Guide

Backpropagation (Backward Propagation of Errors) යනු Neural Networks train කරන්න use කරන **core algorithm** එකයි. මේ notebook එකෙන් අපි step by step backpropagation එක explain කරනවා — math සහිතව, code සහිතව.

---

## Table of Contents

1. [What is a Derivative?](#1-what-is-a-derivative)
2. [What is Backpropagation?](#2-what-is-backpropagation)
3. [Why Do We Need It?](#3-why-do-we-need-it)
4. [Prerequisites — Chain Rule](#4-prerequisites--chain-rule)
5. [Neural Network Architecture](#5-neural-network-architecture)
6. [Step 1: Forward Pass](#6-step-1-forward-pass)
7. [Step 2: Compute Loss](#7-step-2-compute-loss)
8. [Step 3: Backward Pass (Backpropagation)](#8-step-3-backward-pass-backpropagation)
9. [Step 4: Update Weights (Gradient Descent)](#9-step-4-update-weights-gradient-descent)
10. [Full Implementation from Scratch](#10-full-implementation-from-scratch)
11. [Training Loop & Visualization](#11-training-loop--visualization)
12. [Summary](#12-summary)

---

## 1. What is a Derivative? (Derivative කියන්නේ මොකද?)

Backpropagation ඉගෙන ගන්න කලින්, **Derivative** කියන්නේ මොකද කියලා හරියට තේරුම් ගන්න ඕනේ. මේක Deep Learning එකේ **foundational concept** එක.

### Simple Definition:

> **Derivative** කියන්නේ function එකක **rate of change** එක — එනම්, input එක ටිකක් change කළාම output එක **කොපමණ ප්රමාණයකින්** change වෙනවාද කියන එක.

### Real-World Analogy — කන්ද උඩ යන එක:

හිතන්න ඔයා කන්දක් උඩ walk කරනවා:

```
 Height
 ↑ __
 │ / \ steep = large derivative
 │ / \ flat = small derivative
 │ / \__ going down = negative derivative
 │ / \
 │ / \
 │ / \
 └──────────────────→ Distance walked
```

- **Steep uphill** (ලොකු නැග්ම) → Derivative **positive and large** (ඉහළට, වේගයෙන්)
- **Flat ground** (පැතලි) → Derivative **≈ 0** (height change වෙන්නේ නැහැ)
- **Downhill** (බැස්ම) → Derivative **negative** (පහළට යනවා)

### Mathematical Definition:

Function $f(x)$ එකක derivative:

$$f'(x) = \frac{df}{dx} = \lim_{h \to 0} \frac{f(x + h) - f(x)}{h}$$

**In words:** $x$ එක ඉතා කුඩා ප්රමාණයකින් ($h$) change කළාම, $f(x)$ කොපමණ change වෙනවාද?

### Common Derivative Rules:

| Function $f(x)$ | Derivative $f'(x)$ | Example |
|------------------|--------------------|---------|
| $c$ (constant) | $0$ | $f(x) = 5$ → $f'(x) = 0$ |
| $x^n$ (power) | $n \cdot x^{n-1}$ | $f(x) = x^3$ → $f'(x) = 3x^2$ |
| $e^x$ | $e^x$ | $f(x) = e^x$ → $f'(x) = e^x$ |
| $\ln(x)$ | $\frac{1}{x}$ | $f(x) = \ln(x)$ → $f'(x) = \frac{1}{x}$ |
| $\sin(x)$ | $\cos(x)$ | $f(x) = \sin(x)$ → $f'(x) = \cos(x)$ |
| $a \cdot f(x)$ | $a \cdot f'(x)$ | $f(x) = 3x^2$ → $f'(x) = 6x$ |
| $f(x) + g(x)$ | $f'(x) + g'(x)$ | Sum rule |
| $f(x) \cdot g(x)$ | $f'g + fg'$ | Product rule |

### Derivative vs Gradient vs Partial Derivative:

| Term | When to use | Notation |
|------|------------|----------|
| **Derivative** | Variable එකක් විතරයි (single variable) | $\frac{df}{dx}$ or $f'(x)$ |
| **Partial Derivative** | Variables කීපයක් (multi-variable), එක variable එකකට respect to | $\frac{\partial f}{\partial x}$ |
| **Gradient** | හැම partial derivatives එකම vector එකකට | $\nabla f = \left[\frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, ...\right]$ |

### Deep Learning එකේ Derivative එක Important ඇයි?

Neural Network එකක:
- **Loss function** එක minimize කරන්න ඕනේ
- Loss එක minimize කරන්න, **weight එක change කළාම loss එක කොපමණ change වෙනවාද** දැනගන්න ඕනේ
- ඒක තමයි **$\frac{\partial L}{\partial w}$** — Loss w.r.t. Weight එකේ **partial derivative** (gradient)
- මේ gradient එක use කරලා weight update කරනවා: $w_{new} = w_{old} - \eta \cdot \frac{\partial L}{\partial w}$

```
 Derivative tells you:
 ┌────────────────────────────────────────────────────┐
 │ "weight ටිකක් increase කළාම loss increase │
 │ වෙනවා නම් (positive derivative), weight │
 │ decrease කරන්න!" │
 │ │
 │ "weight ටිකක් increase කළාම loss decrease │
 │ වෙනවා නම් (negative derivative), weight │
 │ increase කරන්න!" │
 └────────────────────────────────────────────────────┘
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Derivative Visualization — Step by Step
# ============================================================

# --- Example: f(x) = x² ---
def f(x):
 return x**2

def f_derivative(x):
 """Analytical derivative: f'(x) = 2x"""
 return 2 * x

def numerical_derivative(func, x, h=1e-7):
 """Numerical derivative: lim (f(x+h) - f(x)) / h"""
 return (func(x + h) - func(x - h)) / (2 * h)


# ── Visualize the function and its derivative ──
x_vals = np.linspace(-4, 4, 300)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: f(x) = x² with tangent lines
axes[0].plot(x_vals, f(x_vals), color='#6366f1', linewidth=2.5, label='f(x) = x²')

# Draw tangent lines at specific points
for x0, color in [(-3, '#ef4444'), (0, '#22c55e'), (2, '#f59e0b')]:
 slope = f_derivative(x0)
 y0 = f(x0)
 tangent_x = np.linspace(x0 - 1.5, x0 + 1.5, 50)
 tangent_y = slope * (tangent_x - x0) + y0
 axes[0].plot(tangent_x, tangent_y, color=color, linewidth=2, linestyle='--',
 label=f'Tangent at x={x0}, slope={slope}')
 axes[0].scatter([x0], [y0], color=color, s=100, zorder=5, edgecolors='white', linewidths=2)

axes[0].set_title('f(x) = x² with Tangent Lines', fontsize=13, fontweight='bold')
axes[0].set_xlabel('x')
axes[0].set_ylabel('f(x)')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-2, 16)

# Plot 2: The derivative f'(x) = 2x
axes[1].plot(x_vals, f_derivative(x_vals), color='#f43f5e', linewidth=2.5, label="f'(x) = 2x")
axes[1].axhline(y=0, color='gray', linewidth=1)
axes[1].fill_between(x_vals, f_derivative(x_vals), 0, 
 where=f_derivative(x_vals) > 0, alpha=0.15, color='#22c55e', label='Positive (increasing)')
axes[1].fill_between(x_vals, f_derivative(x_vals), 0,
 where=f_derivative(x_vals) < 0, alpha=0.15, color='#ef4444', label='Negative (decreasing)')
axes[1].set_title("f'(x) = 2x (Derivative)", fontsize=13, fontweight='bold')
axes[1].set_xlabel('x')
axes[1].set_ylabel("f'(x)")
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

# Plot 3: Numerical vs Analytical
analytical = f_derivative(x_vals)
numerical = np.array([numerical_derivative(f, x) for x in x_vals])

axes[2].plot(x_vals, analytical, color='#6366f1', linewidth=2.5, label='Analytical: 2x')
axes[2].plot(x_vals, numerical, color='#f43f5e', linewidth=2, linestyle='--', label='Numerical: Δf/Δx')
axes[2].set_title('Analytical vs Numerical Derivative', fontsize=13, fontweight='bold')
axes[2].set_xlabel('x')
axes[2].set_ylabel("f'(x)")
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle(' Understanding Derivatives — The Foundation of Backpropagation', 
 fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(" Tangent line එකේ slope එක = derivative!")
print(" x = -3 දී slope = -6 (function decreasing)")
print(" x = 0 දී slope = 0 (minimum point!)")
print(" x = 2 දී slope = 4 (function increasing)")

In [ ]:
# ============================================================
# More Derivative Examples — Step by Step
# ============================================================

print("=" * 65)
print(" DERIVATIVE EXAMPLES — Step by Step")
print("=" * 65)

# Example 1: f(x) = x³
print("\n Example 1: f(x) = x³")
print(" ─────────────────────")
print(" Rule: Power Rule → f'(x) = n × x^(n-1)")
print(" f'(x) = 3 × x^(3-1) = 3x²")
print(f" f'(2) = 3 × 2² = 3 × 4 = {3 * 2**2}")
print(f" Numerical check: {numerical_derivative(lambda x: x**3, 2):.6f} ")

# Example 2: f(x) = 3x² + 2x + 1
print("\n Example 2: f(x) = 3x² + 2x + 1")
print(" ──────────────────────────────────")
print(" Rule: Sum Rule + Power Rule")
print(" f'(x) = 3×2x + 2×1 + 0 = 6x + 2")
print(f" f'(3) = 6×3 + 2 = {6*3 + 2}")
print(f" Numerical: {numerical_derivative(lambda x: 3*x**2 + 2*x + 1, 3):.6f} ")

# Example 3: f(x) = e^(2x) — needs chain rule!
print("\n Example 3: f(x) = e^(2x) ← මේකට Chain Rule ඕනේ!")
print(" ─────────────────────────────────────────────────")
print(" Outer: e^u → derivative = e^u")
print(" Inner: u = 2x → derivative = 2")
print(" f'(x) = e^(2x) × 2 = 2e^(2x)")
print(f" f'(1) = 2 × e² = 2 × {np.exp(2):.4f} = {2 * np.exp(2):.4f}")
print(f" Numerical: {numerical_derivative(lambda x: np.exp(2*x), 1):.4f} ")

# Example 4: Partial Derivative
print("\n Example 4: Partial Derivative — f(x, y) = x²y + 3xy²")
print(" ──────────────────────────────────────────────────────────")
print(" ∂f/∂x = 2xy + 3y² (y treat as constant)")
print(" ∂f/∂y = x² + 6xy (x treat as constant)")
x_val, y_val = 2, 3
df_dx = 2*x_val*y_val + 3*y_val**2
df_dy = x_val**2 + 6*x_val*y_val
print(f" At (x=2, y=3): ∂f/∂x = 2×2×3 + 3×9 = {df_dx}")
print(f" ∂f/∂y = 4 + 6×2×3 = {df_dy}")
print(f" Gradient = [{df_dx}, {df_dy}]")

# Numerical verification for partials
h = 1e-7
f_xy = lambda x, y: x**2 * y + 3 * x * y**2
df_dx_num = (f_xy(x_val + h, y_val) - f_xy(x_val - h, y_val)) / (2 * h)
df_dy_num = (f_xy(x_val, y_val + h) - f_xy(x_val, y_val - h)) / (2 * h)
print(f" Numerical: ∂f/∂x = {df_dx_num:.4f}, ∂f/∂y = {df_dy_num:.4f} ")

print("\n" + "=" * 65)
print(" KEY TAKEAWAY:")
print(" Derivative = \"මේ variable එක ටිකක් change කළාම")
print(" output එක කොපමණ change වෙනවාද?\"")
print(" Backpropagation එකේදී: ∂Loss/∂Weight ගණනය කරනවා")
print(" ─ Weight change කළාම Loss කොපමණ change වෙනවාද?")
print("=" * 65)

---

## 2. What is Backpropagation?

Backpropagation යනු Neural Network එකක **weights** සහ **biases** update කරන්න, **loss function** එකේ **gradient** (derivative) calculate කරන algorithm එකයි.

**Simple Intuition:**

1. Network එකට input එකක් දෙනවා → output එකක් ලැබෙනවා (Forward Pass)
2. Output එක actual answer එකත් compare කරනවා → Error/Loss ගණනය කරනවා
3. ඒ error එක ආපහු (backward) propagate කරනවා, හැම weight එකක්ම කොපමණ error එකට contribute කළාද බලනවා
4. Contribute කළ ප්රමාණයට weights adjust කරනවා

```
Input → [Forward Pass] → Output → [Loss] → [Backward Pass] → Updated Weights
 ↑ |
 └────────────────────── Repeat ──────────────────────────────────────┘
```

---

## 3. Why Do We Need It?

Neural Network එකකට අපි randomly initialize කරපු weights තියෙනවා. ඒ weights use කරලා predict කරන answer එක වැරදියි.

**Backpropagation අපිට කියනවා:**
- **කුමන direction** එකකට weight change කරන්න ඕනේද (increase or decrease)
- **කොපමණ ප්රමාණයකින්** change කරන්න ඕනේද

මේක නැතිව neural network එකකට learn කරන්න බැහැ!

## 4. Prerequisites — Chain Rule (දාම නීතිය)

Backpropagation එකේ 'Heart' (හදවත) එක තමයි **Chain Rule** එක. Calculus වලින් ආපු මේ rule එක nested (එකක් ඇතුළේ තව එකක් තියෙන) functions වල derivatives ගන්න use කරනවා.

### The Gear Analogy (ගියර් පද්ධතියක් ගැන හිතන්න):

හිතන්න එකිනෙකට සම්බන්ධ කරපු ගියර් 3ක් තියෙනවා: A, B සහ C.
- ගියර් A එක වටයක් කැරකෙනකොට, B ගියරය වට 2ක් කැරකෙනවා. (Speed ratio = 2)
- ගියර් B එක වටයක් කැරකෙනකොට, C ගියරය වට 3ක් කැරකෙනවා. (Speed ratio = 3)

එතකොට ගියර් A එක වටයක් කැරකෙනකොට, **C ගියරය වට කීයක් කැරකෙනවාද?**
පිළිතුර: $2 \times 3 = 6$ වටයක්! 

මේක තමයි Chain Rule එක! අපි ගුණ කරනවා (Multiply).
$\frac{\text{Change in C}}{\text{Change in A}} = \frac{\text{Change in C}}{\text{Change in B}} \times \frac{\text{Change in B}}{\text{Change in A}}$

### Mathematical Chain Rule:

If $y = f(g(x))$, then:

$$\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$$

### Example:

$$y = (3x + 2)^2$$

Let $u = 3x + 2$, then $y = u^2$

$$\frac{dy}{dx} = \frac{dy}{du} \cdot \frac{du}{dx} = 2u \cdot 3 = 6(3x + 2)$$

Deep Learning වලදී, අන්තිම Loss එකේ ඉඳන් මුල් Input එක දක්වා (Output → Hidden Layer → Input) තියෙන්නේ එකිනෙකට connect වුණ layers. 
ඒක හරියට අර ගියර් වගේ. එක layer එකක් change වුණාම අනිත් එක කොහොම change වෙනවාද බලන්න අපි මේ Chain rule එක පාවිච්චි කරලා ගුණ කරගෙන ගුණ කරගෙන පස්සට (backward) එනවා. එකයි මේකට **"Back-propagation"** කියන්නේ!

---

## 5. Neural Network Architecture

අපි simple network එකක් ගමු — **2 inputs, 1 hidden layer (2 neurons), 1 output**:

```
 Input Layer Hidden Layer Output Layer
 ───────── ──────────── ────────────
 
 x₁ ─────┐ ┌──── h₁ ────┐
 ├────┤ ├───── ŷ (output)
 x₂ ─────┘ └──── h₂ ────┘
 
 Weights: Weights:
 w₁, w₂, w₃, w₄ w₅, w₆
 Biases: b₁, b₂ Bias: b₃
```

### Notation:

| Symbol | Meaning |
|--------|--------|
| $x_i$ | Input features |
| $w_i$ | Weights |
| $b_i$ | Biases |
| $z_i$ | Weighted sum (before activation) |
| $a_i$ | Activation output (after activation function) |
| $\hat{y}$ | Predicted output |
| $y$ | Actual target value |
| $L$ | Loss |
| $\sigma$ | Sigmoid activation function |

---

## 6. Step 1: Forward Pass

Forward pass එකේදී input එක layer by layer pass කරලා output එකක් generate කරනවා.

### Hidden Layer:

$$z_1 = w_1 \cdot x_1 + w_3 \cdot x_2 + b_1$$
$$a_1 = \sigma(z_1)$$

$$z_2 = w_2 \cdot x_1 + w_4 \cdot x_2 + b_2$$
$$a_2 = \sigma(z_2)$$

### Output Layer:

$$z_3 = w_5 \cdot a_1 + w_6 \cdot a_2 + b_3$$
$$\hat{y} = \sigma(z_3)$$

### Sigmoid Activation Function:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

**Sigmoid එකේ ලස්සන property:** 

$$\sigma'(z) = \sigma(z) \cdot (1 - \sigma(z))$$

මේ property එක backpropagation වලදී ගොඩක් important!

In [ ]:
# --- Sigmoid function and its derivative ---
def sigmoid(z):
 """Sigmoid activation function: σ(z) = 1 / (1 + e^(-z))"""
 return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
 """Sigmoid derivative: σ'(z) = σ(z) * (1 - σ(z))"""
 s = sigmoid(z)
 return s * (1 - s)

# --- Visualize Sigmoid and its Derivative ---
z_values = np.linspace(-10, 10, 200)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sigmoid
axes[0].plot(z_values, sigmoid(z_values), color='#6366f1', linewidth=2.5)
axes[0].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Sigmoid Function σ(z)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('z')
axes[0].set_ylabel('σ(z)')
axes[0].grid(True, alpha=0.3)

# Sigmoid Derivative
axes[1].plot(z_values, sigmoid_derivative(z_values), color='#f43f5e', linewidth=2.5)
axes[1].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[1].set_title("Sigmoid Derivative σ'(z)", fontsize=14, fontweight='bold')
axes[1].set_xlabel('z')
axes[1].set_ylabel("σ'(z)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(" Sigmoid output is always between 0 and 1")
print(" Derivative is maximum (0.25) at z = 0")
print(" Derivative approaches 0 for very large/small z (vanishing gradient problem!)")

In [ ]:
# ============================================================
# FORWARD PASS — Numerical Example
# ============================================================

# Inputs
x1, x2 = 0.05, 0.10

# Target output
y_true = 0.01

# --- Initialize Weights (random but fixed for reproducibility) ---
w1, w2 = 0.15, 0.25
w3, w4 = 0.20, 0.30
w5, w6 = 0.40, 0.50

# --- Initialize Biases ---
b1, b2 = 0.35, 0.35
b3 = 0.60

print("=" * 60)
print(" FORWARD PASS")
print("=" * 60)

# --- Hidden Layer ---
z1 = w1 * x1 + w3 * x2 + b1
a1 = sigmoid(z1)

z2 = w2 * x1 + w4 * x2 + b2
a2 = sigmoid(z2)

print(f"\n Hidden Layer:")
print(f" z1 = {w1}×{x1} + {w3}×{x2} + {b1} = {z1:.6f}")
print(f" a1 = σ(z1) = σ({z1:.6f}) = {a1:.6f}")
print(f"")
print(f" z2 = {w2}×{x1} + {w4}×{x2} + {b2} = {z2:.6f}")
print(f" a2 = σ(z2) = σ({z2:.6f}) = {a2:.6f}")

# --- Output Layer ---
z3 = w5 * a1 + w6 * a2 + b3
y_pred = sigmoid(z3)

print(f"\n Output Layer:")
print(f" z3 = {w5}×{a1:.6f} + {w6}×{a2:.6f} + {b3} = {z3:.6f}")
print(f" ŷ = σ(z3) = σ({z3:.6f}) = {y_pred:.6f}")
print(f"\n Target y = {y_true}")
print(f" Prediction ŷ = {y_pred:.6f}")
print(f" Error = {abs(y_true - y_pred):.6f}")

---

## 7. Step 2: Compute Loss

Prediction එක actual value එකෙන් කොතරම් දුරද measure කරන්න **Loss Function** එකක් use කරනවා.

### Mean Squared Error (MSE) Loss:

$$L = \frac{1}{2}(y - \hat{y})^2$$

> **Note:** $\frac{1}{2}$ එක ගන්නේ derivative ගත්තම simplify වෙන්න.

### Binary Cross-Entropy Loss (for classification):

$$L = -[y \cdot \log(\hat{y}) + (1-y) \cdot \log(1-\hat{y})]$$

අපි මේ example එකේදී **MSE** use කරමු.

In [ ]:
# ============================================================
# COMPUTE LOSS
# ============================================================

loss = 0.5 * (y_true - y_pred) ** 2

print("=" * 60)
print(" LOSS COMPUTATION")
print("=" * 60)
print(f"\n L = ½ × (y - ŷ)²")
print(f" L = ½ × ({y_true} - {y_pred:.6f})²")
print(f" L = ½ × ({y_true - y_pred:.6f})²")
print(f" L = ½ × {(y_true - y_pred)**2:.6f}")
print(f" L = {loss:.6f}")
print(f"\n මේ loss value එක minimize කරන්න තමයි weights update කරන්නේ!")

## 8. Step 3: Backward Pass (Backpropagation)

මෙතනින් තමයි actual backpropagation එක පටන් ගන්නේ! 

**Goal:** හැම weight එකකටම Loss එකේ gradient (derivative) ගණනය කරනවා. (ඒ කියන්නේ Weight එක පොඩ්ඩක් වෙනස් කළොත්, Loss එක කොච්චරකින් වෙනස් වෙනවද කියන එක හොයනවා).

$$\frac{\partial L}{\partial w_i} = \text{?}$$

### The "Blame Game" Analogy (වැරැද්ද කාගේද?):

හිතන්න ලොකු කර්මාන්තශාලාවක් (factory) තියෙනවා. 
- අන්තිමට හදපු product එකේ පොඩි වැරැද්දක් (Loss) තියෙනවා.
- Boss (Loss Function) අහනවා "කවුද මේ වැරැද්ද කළේ?" කියලා අන්තිමට හිටපු Supervisor (Output Layer) ගෙන්.
- Supervisor කියනවා "මම විතරක් නෙවෙයි, මට බඩු එවපු කට්ටියත් වැරදියි" කියලා, එයාට බඩු එවපු Worker 1 සහ Worker 2 (Hidden Layer) ට ඒ වැරැද්දෙන් කොටසක් පවරනවා.
- මේ විදියට, අන්තිම වැරැද්ද (Final Error/Loss) ඒකට දායක වුණු හැම කෙනාටම (Weights වලට) බෙදී යනවා. මේ බෙදී යන Error එකට අපි කියනවා **$\delta$ (Delta - Error Signal)** කියලා.

### Chain Rule Application — Output Layer Weights

#### For $w_5$ (අවසාන ලේයරයේ weight එක):

$$\frac{\partial L}{\partial w_5} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_3} \cdot \frac{\partial z_3}{\partial w_5}$$

Let's compute each part (ගියර් තුනක් වගේ):

**Part 1: $\frac{\partial L}{\partial \hat{y}} = -(y - \hat{y})$**
(අපේ prediction එක ඇත්ත උත්තරෙන් කොච්චර ඈතද?)

**Part 2: $\frac{\partial \hat{y}}{\partial z_3} = \hat{y}(1 - \hat{y})$**
(Sigmoid function එක කොච්චර වේගයෙන් වෙනස් වෙනවද?)

**Part 3: $\frac{\partial z_3}{\partial w_5} = a_1$**
(මේ weight එකට ආපු input එක කොච්චර ලොකුද? Input එක ලොකු නම්, weight එකේ වැරැද්දත් ලොකුයි!)

Therefore:

$$\frac{\partial L}{\partial w_5} = -(y - \hat{y}) \cdot \hat{y}(1 - \hat{y}) \cdot a_1$$

### Defining $\delta$ (Error Signal / Blame):

අපි මුල් කොටස් දෙක එකතු කරලා **$\delta_{output}$** (අවසාන ලේයරයේ වැරැද්ද/Blame එක) හදනවා:

$$\delta_{output} = -(y - \hat{y}) \cdot \hat{y}(1 - \hat{y})$$

This simplifies our notation (Gradient එක = Error × Input):

$$\frac{\partial L}{\partial w_5} = \delta_{output} \cdot a_1$$
$$\frac{\partial L}{\partial w_6} = \delta_{output} \cdot a_2$$

---

### Chain Rule Application — Hidden Layer Weights (Workers ලගේ වැරැද්ද)

Hidden layer weights වල gradients හොයන්න, අර අවසාන Supervisor ගේ වැරැද්ද ($\delta_{output}$) backward pass කරන්න ඕනේ.

#### For $w_1$:

$$\frac{\partial L}{\partial w_1} = \frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_3} \cdot \frac{\partial z_3}{\partial a_1} \cdot \frac{\partial a_1}{\partial z_1} \cdot \frac{\partial z_1}{\partial w_1}$$

Breaking it down (දිග Chain එකක්!):

- $\frac{\partial L}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z_3} = \delta_{output}$ (අවසාන Error එක)
- $\frac{\partial z_3}{\partial a_1} = w_5$ (අවසාන Error එකෙන් මේ hidden neuron එකට එන බලපෑම - Connection strength එක)
- $\frac{\partial a_1}{\partial z_1} = a_1(1 - a_1)$ (මේ neuron එකේ sigmoid derivative එක)
- $\frac{\partial z_1}{\partial w_1} = x_1$ (මේ weight එකට ආපු input එක)

**Hidden layer delta (Hidden neuron එකේ Blame එක):**

$$\delta_{h1} = (\delta_{output} \cdot w_5) \cdot a_1(1 - a_1)$$

ඒ කියන්නේ: `Hidden Error = (Output Error × Weight connecting them) × Activation Derivative`

$$\frac{\partial L}{\partial w_1} = \delta_{h1} \cdot x_1$$

Similarly for other hidden weights:

$$\delta_{h2} = (\delta_{output} \cdot w_6) \cdot a_2(1 - a_2)$$

In [ ]:
# ============================================================
# BACKWARD PASS — Step by Step Gradient Computation
# ============================================================

print("=" * 60)
print(" BACKWARD PASS (BACKPROPAGATION)")
print("=" * 60)

# ── Step 3a: Output Layer Error Signal (δ_output) ──
dL_dy_pred = -(y_true - y_pred) # ∂L/∂ŷ
dy_pred_dz3 = y_pred * (1 - y_pred) # ∂ŷ/∂z3 (sigmoid derivative)
delta_output = dL_dy_pred * dy_pred_dz3 # δ_output

print(f"\n Output Layer Error Signal:")
print(f" ∂L/∂ŷ = -(y - ŷ) = -({y_true} - {y_pred:.6f}) = {dL_dy_pred:.6f}")
print(f" ∂ŷ/∂z3 = ŷ(1-ŷ) = {y_pred:.6f} × {1-y_pred:.6f} = {dy_pred_dz3:.6f}")
print(f" δ_output = ∂L/∂ŷ × ∂ŷ/∂z3 = {dL_dy_pred:.6f} × {dy_pred_dz3:.6f} = {delta_output:.6f}")

# ── Step 3b: Gradients for Output Layer Weights ──
dL_dw5 = delta_output * a1
dL_dw6 = delta_output * a2
dL_db3 = delta_output * 1 # bias gradient

print(f"\n Output Layer Weight Gradients:")
print(f" ∂L/∂w5 = δ_output × a1 = {delta_output:.6f} × {a1:.6f} = {dL_dw5:.6f}")
print(f" ∂L/∂w6 = δ_output × a2 = {delta_output:.6f} × {a2:.6f} = {dL_dw6:.6f}")
print(f" ∂L/∂b3 = δ_output × 1 = {dL_db3:.6f}")

# ── Step 3c: Hidden Layer Error Signals ──
delta_h1 = delta_output * w5 * (a1 * (1 - a1))
delta_h2 = delta_output * w6 * (a2 * (1 - a2))

print(f"\n Hidden Layer Error Signals:")
print(f" δ_h1 = δ_output × w5 × a1(1-a1)")
print(f" = {delta_output:.6f} × {w5} × {a1:.6f} × {1-a1:.6f}")
print(f" = {delta_h1:.6f}")
print(f"")
print(f" δ_h2 = δ_output × w6 × a2(1-a2)")
print(f" = {delta_output:.6f} × {w6} × {a2:.6f} × {1-a2:.6f}")
print(f" = {delta_h2:.6f}")

# ── Step 3d: Gradients for Hidden Layer Weights ──
dL_dw1 = delta_h1 * x1
dL_dw2 = delta_h2 * x1
dL_dw3 = delta_h1 * x2
dL_dw4 = delta_h2 * x2
dL_db1 = delta_h1 * 1
dL_db2 = delta_h2 * 1

print(f"\n Hidden Layer Weight Gradients:")
print(f" ∂L/∂w1 = δ_h1 × x1 = {delta_h1:.6f} × {x1} = {dL_dw1:.8f}")
print(f" ∂L/∂w2 = δ_h2 × x1 = {delta_h2:.6f} × {x1} = {dL_dw2:.8f}")
print(f" ∂L/∂w3 = δ_h1 × x2 = {delta_h1:.6f} × {x2} = {dL_dw3:.8f}")
print(f" ∂L/∂w4 = δ_h2 × x2 = {delta_h2:.6f} × {x2} = {dL_dw4:.8f}")
print(f" ∂L/∂b1 = δ_h1 = {dL_db1:.8f}")
print(f" ∂L/∂b2 = δ_h2 = {dL_db2:.8f}")

# ── Summary Table ──
print(f"\n{'='*60}")
print(f" GRADIENT SUMMARY")
print(f"{'='*60}")
print(f" {'Weight':<8} {'Value':<10} {'Gradient':<15}")
print(f" {'─'*35}")
print(f" {'w1':<8} {w1:<10} {dL_dw1:<15.8f}")
print(f" {'w2':<8} {w2:<10} {dL_dw2:<15.8f}")
print(f" {'w3':<8} {w3:<10} {dL_dw3:<15.8f}")
print(f" {'w4':<8} {w4:<10} {dL_dw4:<15.8f}")
print(f" {'w5':<8} {w5:<10} {dL_dw5:<15.8f}")
print(f" {'w6':<8} {w6:<10} {dL_dw6:<15.8f}")
print(f" {'b1':<8} {b1:<10} {dL_db1:<15.8f}")
print(f" {'b2':<8} {b2:<10} {dL_db2:<15.8f}")
print(f" {'b3':<8} {b3:<10} {dL_db3:<15.8f}")

## 9. Step 4: Update Weights (Gradient Descent)

Gradients calculate කරාට පස්සේ (ඒ කියන්නේ කන්දේ බෑවුම හොයාගත්තට පස්සේ), weights update කරනවා **Gradient Descent** use කරලා. 
අපේ අරමුණ Loss එක අවම වෙන තැනට (කන්දේ පහළටම) යන එකයි.

$$w_{new} = w_{old} - \eta \cdot \frac{\partial L}{\partial w}$$

- අපි **අඩු කරන්නේ (Subtract)** ඇයි? Gradient එකෙන් පෙන්නන්නේ Loss එක *වැඩි වෙන* පැත්ත. ඒක නිසා අපි Gradient එකේ විරුද්ධ පැත්තට (-) යන්න ඕනේ Loss එක අඩු කරගන්න.
- $\eta$ (eta) කියන්නේ **Learning Rate (ඉගෙනීමේ වේගය)** — අපි කන්ද පහළට තියන අඩි වල ප්රමාණය (Step size).

### Learning Rate (η) තෝරාගැනීම:

- **Too large (ලොකු වැඩියි):** කන්දෙන් පල්ලෙහාට පනින්න ගිහින් අනිත් පැත්තට විසිවෙනවා (Overshoots). Loss එක උඩ පහළ යනවා (Oscillates).
- **Too small (කුඩා වැඩියි):** කුහුඹුවෙක් වගේ යනවා. පහළට යන්න ගොඩක් කල් යනවා (Very slow learning).
- **Just right (හරියටම ගැලපෙනවා):** ලස්සනට, ඉක්මනින් පහළට යනවා (Smooth convergence).

```
 Loss
 ↑
 │\ Too large η (ලොකු අඩි තියනවා)
 │ \ / ↗ oscillates
 │ \ / ↗
 │ \/↗
 │ \ Good η (හරියට යනවා)
 │ \___________ ← converges
 │
 │ Too small η (ගොළුබෙල්ලෙක් වගේ)
 │ ________________________________ ← barely moves
 └──────────────────────────────→ Epochs
```

In [ ]:
# ============================================================
# WEIGHT UPDATE — Gradient Descent
# ============================================================

learning_rate = 0.5 # η

print("=" * 60)
print(" WEIGHT UPDATE (Gradient Descent)")
print("=" * 60)
print(f"\n Learning Rate (η) = {learning_rate}")
print(f" Rule: w_new = w_old - η × ∂L/∂w\n")

# Calculate new weights
w1_new = w1 - learning_rate * dL_dw1
w2_new = w2 - learning_rate * dL_dw2
w3_new = w3 - learning_rate * dL_dw3
w4_new = w4 - learning_rate * dL_dw4
w5_new = w5 - learning_rate * dL_dw5
w6_new = w6 - learning_rate * dL_dw6
b1_new = b1 - learning_rate * dL_db1
b2_new = b2 - learning_rate * dL_db2
b3_new = b3 - learning_rate * dL_db3

print(f" {'Param':<6} {'Old Value':<12} {'Gradient':<14} {'New Value':<12} {'Change':<12}")
print(f" {'─'*58}")

params = [
 ('w1', w1, dL_dw1, w1_new),
 ('w2', w2, dL_dw2, w2_new),
 ('w3', w3, dL_dw3, w3_new),
 ('w4', w4, dL_dw4, w4_new),
 ('w5', w5, dL_dw5, w5_new),
 ('w6', w6, dL_dw6, w6_new),
 ('b1', b1, dL_db1, b1_new),
 ('b2', b2, dL_db2, b2_new),
 ('b3', b3, dL_db3, b3_new),
]

for name, old, grad, new in params:
 change = new - old
 arrow = '↑' if change > 0 else '↓'
 print(f" {name:<6} {old:<12.6f} {grad:<14.8f} {new:<12.6f} {change:+.8f} {arrow}")

# Verify: Forward pass with new weights
print(f"\n{'='*60}")
print(" VERIFICATION: Forward Pass with Updated Weights")
print(f"{'='*60}")

z1_new = w1_new * x1 + w3_new * x2 + b1_new
a1_new = sigmoid(z1_new)
z2_new = w2_new * x1 + w4_new * x2 + b2_new
a2_new = sigmoid(z2_new)
z3_new = w5_new * a1_new + w6_new * a2_new + b3_new
y_pred_new = sigmoid(z3_new)
loss_new = 0.5 * (y_true - y_pred_new) ** 2

print(f"\n Before update: ŷ = {y_pred:.6f}, Loss = {loss:.6f}")
print(f" After update: ŷ = {y_pred_new:.6f}, Loss = {loss_new:.6f}")
print(f" Loss decreased by: {loss - loss_new:.6f} ")
print(f"\n Loss reduced! Network එක learn කරනවා!")

## Full Implementation Breakdown (සම්පූර්ණ කේතය පියවරෙන් පියවර)

දැන් අපි ඉහත ඉගෙන ගත්තු සියලුම දේවල් එකතු කරලා සම්පූර්ණ Neural Network එකක් හදමු. මෙහිදී අපි Object Oriented Programming (OOP) භාවිතයෙන් `NeuralNetwork` කියන class එක නිර්මාණය කරනවා. 

මෙහි ප්රධාන කොටස් 4ක් තියෙනවා:
1. **`__init__` (Initialization):** මුලින්ම ජාලයේ බර (weights) සහ biases අහඹු ලෙස (randomly) සකස් කිරීම. අපි මෙහිදී කුඩා අගයන් ලබා දෙනවා.
2. **`forward` (Forward Pass):** Input එක ලබා දුන් විට, එය ස්ථර (layers) හරහා ගොස් අවසාන පිළිතුර (prediction) ලබා දෙන පියවර. මෙහිදී Sigmoid activation එක භාවිතා වෙනවා.
3. **`backward` (Backward Pass / Backpropagation):** අපි කලින් කතා කරපු Chain Rule එක පාවිච්චි කරලා, අවසාන වැරැද්ද (Loss) මුලට ගෙන එමින් එක් එක් weight එකේ Gradient එක ගණනය කරන පියවර.
4. **`update_weights` (Gradient Descent):** ගණනය කරපු Gradients භාවිතයෙන් Weights සහ Biases අලුත් කිරීම.

පහත කේතය කියවන විට, එක් එක් පේළියෙන් කරන්නේ කුමක්ද යන්න පැහැදිලිව තේරුම් ගන්න. සෑම පියවරක්ම අපි ඉහතින් සාකච්ඡා කළ ගණිතමය සමීකරණ වල සෘජු පරිවර්තනයක් (direct translation) වේ.

---

## 10. Full Implementation from Scratch

දැන් අපි complete Neural Network class එකක් build කරමු — Forward Pass, Backward Pass, සහ Training Loop එකත් එක්ක.

### Architecture: XOR Problem Solver

XOR problem එක solve කරන network එකක් build කරමු:

| $x_1$ | $x_2$ | $y$ (XOR) |
|-------|-------|----------|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

> **Note:** XOR problem එක linear model වලින් solve කරන්න බැහැ. Neural Network එකක් ඕනමයි!

In [ ]:
class NeuralNetwork:
 """
 Simple 2-Layer Neural Network built from scratch.
 Architecture: Input(2) → Hidden(4, sigmoid) → Output(1, sigmoid)
 
 This implementation demonstrates every step of backpropagation.
 """
 
 def __init__(self, input_size=2, hidden_size=4, output_size=1, learning_rate=0.5):
 self.lr = learning_rate
 
 # Initialize weights with small random values
 np.random.seed(42)
 self.W1 = np.random.randn(input_size, hidden_size) * 0.5 # (2, 4)
 self.b1 = np.zeros((1, hidden_size)) # (1, 4)
 self.W2 = np.random.randn(hidden_size, output_size) * 0.5 # (4, 1)
 self.b2 = np.zeros((1, output_size)) # (1, 1)
 
 # Store loss history for plotting
 self.loss_history = []
 
 def sigmoid(self, z):
 """Sigmoid activation function"""
 return 1 / (1 + np.exp(-np.clip(z, -500, 500)))
 
 def sigmoid_derivative(self, a):
 """Derivative of sigmoid: σ'(z) = a * (1 - a), where a = σ(z)"""
 return a * (1 - a)
 
 def forward(self, X):
 """
 FORWARD PASS
 
 Step 1: Calculate hidden layer
 z1 = X @ W1 + b1
 a1 = σ(z1)
 
 Step 2: Calculate output layer
 z2 = a1 @ W2 + b2
 a2 = σ(z2)
 """
 # Hidden Layer
 self.z1 = X @ self.W1 + self.b1 # Weighted sum
 self.a1 = self.sigmoid(self.z1) # Activation
 
 # Output Layer
 self.z2 = self.a1 @ self.W2 + self.b2 # Weighted sum
 self.a2 = self.sigmoid(self.z2) # Activation (prediction)
 
 return self.a2
 
 def compute_loss(self, y_true, y_pred):
 """ Mean Squared Error Loss"""
 m = y_true.shape[0] # number of samples
 loss = np.mean(0.5 * (y_true - y_pred) ** 2)
 return loss
 
 def backward(self, X, y_true):
 """
 BACKWARD PASS (Backpropagation)
 
 Computes gradients for all parameters using chain rule.
 
 Step 1: Output layer gradients
 δ_output = -(y - ŷ) × σ'(z2)
 ∂L/∂W2 = a1.T @ δ_output
 ∂L/∂b2 = sum(δ_output)
 
 Step 2: Hidden layer gradients (propagate error backward)
 δ_hidden = (δ_output @ W2.T) × σ'(z1)
 ∂L/∂W1 = X.T @ δ_hidden
 ∂L/∂b1 = sum(δ_hidden)
 """
 m = X.shape[0] # number of samples
 
 # ── Step 1: Output Layer Gradients ──
 # δ_output = dL/da2 * da2/dz2
 # dL/da2 = -(y - ŷ)
 # da2/dz2 = σ'(z2) = a2 * (1 - a2)
 self.delta_output = -(y_true - self.a2) * self.sigmoid_derivative(self.a2)
 
 # Gradients for W2 and b2
 self.dW2 = (1/m) * self.a1.T @ self.delta_output
 self.db2 = (1/m) * np.sum(self.delta_output, axis=0, keepdims=True)
 
 # ── Step 2: Hidden Layer Gradients ──
 # δ_hidden = (δ_output @ W2.T) * σ'(z1)
 # This is where the error gets "backpropagated" through the network!
 self.delta_hidden = (self.delta_output @ self.W2.T) * self.sigmoid_derivative(self.a1)
 
 # Gradients for W1 and b1
 self.dW1 = (1/m) * X.T @ self.delta_hidden
 self.db1 = (1/m) * np.sum(self.delta_hidden, axis=0, keepdims=True)
 
 def update_weights(self):
 """
 GRADIENT DESCENT — Update all parameters
 
 w_new = w_old - η × ∂L/∂w
 """
 self.W1 -= self.lr * self.dW1
 self.b1 -= self.lr * self.db1
 self.W2 -= self.lr * self.dW2
 self.b2 -= self.lr * self.db2
 
 def train(self, X, y, epochs=10000, print_every=1000):
 """
 TRAINING LOOP
 
 For each epoch:
 1. Forward Pass → get predictions
 2. Compute Loss → measure error
 3. Backward Pass → compute gradients
 4. Update Weights → gradient descent
 """
 print("=" * 60)
 print(" TRAINING STARTED")
 print("=" * 60)
 
 for epoch in range(epochs):
 # Step 1: Forward Pass
 y_pred = self.forward(X)
 
 # Step 2: Compute Loss
 loss = self.compute_loss(y, y_pred)
 self.loss_history.append(loss)
 
 # Step 3: Backward Pass
 self.backward(X, y)
 
 # Step 4: Update Weights
 self.update_weights()
 
 # Print progress
 if (epoch + 1) % print_every == 0 or epoch == 0:
 print(f" Epoch {epoch+1:>6}/{epochs} │ Loss: {loss:.6f}")
 
 print(f"\n Training Complete! Final Loss: {self.loss_history[-1]:.6f}")

print(" NeuralNetwork class created successfully!")
print("\nArchitecture: Input(2) → Hidden(4, sigmoid) → Output(1, sigmoid)")

---

## 11. Training Loop & Visualization

දැන් අපි XOR data set එකෙන් network එක train කරලා results visualize කරමු.

In [ ]:
# ============================================================
# XOR Dataset
# ============================================================

X = np.array([
 [0, 0],
 [0, 1],
 [1, 0],
 [1, 1]
])

y = np.array([
 [0],
 [1],
 [1],
 [0]
])

print("XOR Dataset:")
print(f"{'x1':<5} {'x2':<5} {'y (XOR)':<8}")
print("-" * 18)
for i in range(len(X)):
 print(f"{X[i,0]:<5} {X[i,1]:<5} {y[i,0]:<8}")

In [ ]:
# ============================================================
# Train the Network
# ============================================================

nn = NeuralNetwork(input_size=2, hidden_size=4, output_size=1, learning_rate=2.0)
nn.train(X, y, epochs=10000, print_every=1000)

In [ ]:
# ============================================================
# Test Predictions
# ============================================================

print("=" * 60)
print(" PREDICTIONS AFTER TRAINING")
print("=" * 60)

predictions = nn.forward(X)

print(f"\n {'x1':<5} {'x2':<5} {'Target':<10} {'Predicted':<12} {'Rounded':<10} {'/'}")
print(f" {'─'*50}")

all_correct = True
for i in range(len(X)):
 pred = predictions[i, 0]
 rounded = round(pred)
 correct = "" if rounded == y[i, 0] else ""
 if rounded != y[i, 0]:
 all_correct = False
 print(f" {X[i,0]:<5} {X[i,1]:<5} {y[i,0]:<10} {pred:<12.6f} {rounded:<10} {correct}")

if all_correct:
 print(f"\n Network එක XOR problem එක solve කරා! All predictions correct!")
else:
 print(f"\n Some predictions are wrong. Try more epochs or adjust learning rate.")

In [ ]:
# ============================================================
# Visualization — Loss Curve & Decision Boundary
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Plot 1: Loss Curve ──
ax1 = axes[0]
ax1.plot(nn.loss_history, color='#6366f1', linewidth=1.5, alpha=0.8)
ax1.set_title(' Training Loss Over Epochs', fontsize=14, fontweight='bold', pad=15)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss (MSE)', fontsize=12)
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)
ax1.axhline(y=nn.loss_history[-1], color='#f43f5e', linestyle='--', alpha=0.5,
 label=f'Final Loss: {nn.loss_history[-1]:.6f}')
ax1.legend(fontsize=11)

# ── Plot 2: Decision Boundary ──
ax2 = axes[1]

# Create a mesh grid
xx, yy = np.meshgrid(
 np.linspace(-0.5, 1.5, 200),
 np.linspace(-0.5, 1.5, 200)
)
grid = np.c_[xx.ravel(), yy.ravel()]
predictions_grid = nn.forward(grid).reshape(xx.shape)

# Contour plot
contour = ax2.contourf(xx, yy, predictions_grid, levels=50, cmap='RdYlBu_r', alpha=0.8)
plt.colorbar(contour, ax=ax2, label='Predicted Value')

# Plot data points
for i in range(len(X)):
 color = '#22c55e' if y[i, 0] == 1 else '#ef4444'
 marker = 'o' if y[i, 0] == 1 else 's'
 label = f'y={int(y[i,0])}' if i < 2 else None
 ax2.scatter(X[i, 0], X[i, 1], c=color, s=200, edgecolors='white',
 linewidths=2, marker=marker, zorder=5, label=label)

ax2.set_title(' Decision Boundary (XOR)', fontsize=14, fontweight='bold', pad=15)
ax2.set_xlabel('x₁', fontsize=12)
ax2.set_ylabel('x₂', fontsize=12)
ax2.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

print(" Left: Loss decreases over epochs — network is learning!")
print(" Right: Decision boundary separates XOR classes (non-linear boundary!)")

In [ ]:
# ============================================================
# Visualization — Gradient Flow
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap of W1 (input → hidden weights)
im1 = axes[0].imshow(nn.W1, cmap='coolwarm', aspect='auto')
axes[0].set_title('W1: Input → Hidden Weights', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Hidden Neurons')
axes[0].set_ylabel('Input Features')
axes[0].set_xticks(range(nn.W1.shape[1]))
axes[0].set_yticks(range(nn.W1.shape[0]))
axes[0].set_xticklabels([f'h{i+1}' for i in range(nn.W1.shape[1])])
axes[0].set_yticklabels([f'x{i+1}' for i in range(nn.W1.shape[0])])
for i in range(nn.W1.shape[0]):
 for j in range(nn.W1.shape[1]):
 axes[0].text(j, i, f'{nn.W1[i,j]:.2f}', ha='center', va='center', 
 fontsize=11, fontweight='bold', color='black')
plt.colorbar(im1, ax=axes[0])

# Heatmap of W2 (hidden → output weights)
im2 = axes[1].imshow(nn.W2, cmap='coolwarm', aspect='auto')
axes[1].set_title('W2: Hidden → Output Weights', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Output Neurons')
axes[1].set_ylabel('Hidden Neurons')
axes[1].set_xticks(range(nn.W2.shape[1]))
axes[1].set_yticks(range(nn.W2.shape[0]))
axes[1].set_xticklabels([f'ŷ'])
axes[1].set_yticklabels([f'h{i+1}' for i in range(nn.W2.shape[0])])
for i in range(nn.W2.shape[0]):
 for j in range(nn.W2.shape[1]):
 axes[1].text(j, i, f'{nn.W2[i,j]:.2f}', ha='center', va='center',
 fontsize=11, fontweight='bold', color='black')
plt.colorbar(im2, ax=axes[1])

plt.suptitle(' Learned Weight Matrices After Training', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(" Weights have been adjusted by backpropagation to solve XOR!")
print(" Positive weights (red) = excitatory connections")
print(" Negative weights (blue) = inhibitory connections")

---

## 12. Summary

### Backpropagation Algorithm — Complete Overview

```
┌──────────────────────────────────────────────────────────────────┐
│ BACKPROPAGATION ALGORITHM │
├──────────────────────────────────────────────────────────────────┤
│ │
│ Step 1: FORWARD PASS │
│ ───────────────────── │
│ • Input → Hidden: z = Wx + b, a = σ(z) │
│ • Hidden → Output: same process │
│ • Get prediction ŷ │
│ │
│ Step 2: COMPUTE LOSS │
│ ────────────────── │
│ • L = ½(y - ŷ)² (MSE) │
│ • Measures how wrong the prediction is │
│ │
│ Step 3: BACKWARD PASS │
│ ───────────────────── │
│ • Compute δ_output = -(y-ŷ) × σ'(z) │
│ • Compute ∂L/∂W2 = a1.T × δ_output │
│ • Backpropagate: δ_hidden = (δ_output × W2.T) × σ'(z1) │
│ • Compute ∂L/∂W1 = X.T × δ_hidden │
│ • Uses CHAIN RULE throughout │
│ │
│ Step 4: UPDATE WEIGHTS │
│ ────────────────────── │
│ • W = W - η × ∂L/∂W (Gradient Descent) │
│ • η = learning rate │
│ │
│ Step 5: REPEAT until loss is minimized │
│ │
└──────────────────────────────────────────────────────────────────┘
```

### Key Takeaways:

1. **Derivative** — function එකක rate of change; weight change කළාම loss කොපමණ change වෙනවාද measure කරනවා
2. **Forward Pass** — data flows input → output, predictions generate වෙනවා
3. **Loss Function** — prediction එක target එකෙන් කොපමණ වෙනස්ද measure කරනවා
4. **Chain Rule** — nested function derivatives ගන්න use කරනවා (backprop එකේ core)
5. **Backward Pass** — gradients output layer සිට input layer දක්වා compute කරනවා
6. **Gradient Descent** — gradients use කරලා weights update කරනවා, loss minimize වෙන direction එකට
7. **Learning Rate** — step size එක control කරනවා (too big = diverge, too small = slow)

### Common Issues:

| Issue | Cause | Solution |
|-------|-------|----------|
| Vanishing Gradient | Sigmoid squashes gradients | Use ReLU activation |
| Exploding Gradient | Very large weight updates | Gradient clipping, smaller LR |
| Slow Convergence | Learning rate too small | Increase LR, use Adam optimizer |
| Oscillating Loss | Learning rate too large | Decrease LR |
| Stuck in Local Minimum | Poor initialization | Xavier/He initialization, momentum |

### Next Steps:

- **Activation Functions:** ReLU, Leaky ReLU, Tanh
- **Optimizers:** SGD with Momentum, Adam, RMSProp
- **Regularization:** Dropout, L2 Regularization
- **Deep Networks:** Multiple hidden layers
- **Frameworks:** PyTorch, TensorFlow (auto-differentiation!)